# Monitoring part 1: data quality

**Question this notebook answers:** *is the data feeding the forecaster still healthy?*

If the input data breaks (missing records, renamed fields, duplicates, clock bugs) or changes a lot (new chargers,
different usage), the model will produce bad forecasts **without any error message**. So every week, before the
data reaches the model, we run a set of checks and give each one a status:

| Status | Meaning |
|---|---|
| **OK** | normal |
| **WARN** | unusual, shown in the status board, nobody is paged |
| **ALERT** | something is wrong, the on-call engineer is paged |

**No live data?** The ACN files stop in September 2021, so we **replay the test period one week at a time**, as if
each week had just arrived. We do it twice:

1. **Run A, real data:** what the monitor would really have reported.
2. **Run B, injected problems:** we break the Caltech data on purpose, to prove each check catches its problem.

This is the *"validate timestamps, site IDs, and missing intervals"* step of the Milestone 2 pipeline.

## 1. Setup

In [1]:
import glob
import json
import os

import holidays
import numpy as np
import pandas as pd

DATA_DIR = "./acn_dataset"
OUT = "outputs/monitoring"
os.makedirs(OUT, exist_ok=True)

SITES = ["caltech", "jpl"]
TZ = "America/Los_Angeles"
CA_HOLIDAYS = holidays.US(state="CA")
L, H = 672, 16                    # model window: 1 week of history, 4 hours ahead
WEEK = pd.Timedelta(days=7)

# Fields the pipeline cannot work without
REQUIRED = ["sessionID", "spaceID", "connectionTime", "disconnectTime", "kWhDelivered"]
KNOWN = REQUIRED + ["_id", "userInputs", "userID", "stationID", "siteID",
                    "clusterID", "doneChargingTime", "timezone"]

### The checks and their thresholds

All thresholds are in one table so they are easy to read and change.

* `bad = high`: a bigger value is worse. `bad = low`: a smaller value is worse.
* `on alert = page`: something is broken now. `on alert = ticket`: the data has moved away from the
  **training data**. That is a slow, lasting change, so it opens a *retraining ticket* instead of paging every week.

In [2]:
CHECKS = pd.DataFrame([
    # name,             group,          warn,  alert,  bad,    on alert, what it catches
    ("missing_fields",  "Schema",       1,     1,      "high", "page",   "a required field is missing (e.g. renamed upstream)"),
    ("new_fields",      "Schema",       1,     np.inf, "high", "page",   "an unknown field appeared (usually the other half of a rename)"),
    ("null_pct",        "Completeness", 0.5,   2,      "high", "page",   "% of sessions with an empty required field (training: 0%)"),
    ("volume_ratio",    "Completeness", 0.6,   0.3,    "low",  "page",   "sessions this week / median of the last 4 good weeks"),
    ("dead_weekdays",   "Completeness", 1,     2,      "high", "page",   "working days with zero arrivals = ingestion outage"),
    ("invalid_pct",     "Validity",     1,     5,      "high", "page",   "% impossible sessions (ends before start, kWh < 0 or > 100)"),
    ("duplicate_pct",   "Validity",     0.01,  2,      "high", "page",   "% of session IDs received twice"),
    ("new_ports",       "Categories",   1,     5,      "high", "page",   "charging ports never seen before = capacity changed"),
    ("psi_vs_recent",   "Drift",        0.10,  0.25,   "high", "page",   "occupancy distribution vs the last 4 good weeks (sudden change)"),
    ("hour_corr",       "Drift",        0.90,  0.75,   "low",  "page",   "daily occupancy curve vs the last 4 good weeks (clock/timezone bug)"),
    ("psi_vs_train",    "Drift",        0.10,  0.25,   "high", "ticket", "occupancy distribution vs training data (regime change)"),
    ("psi_scale",       "Drift",        0.10,  0.25,   "high", "ticket", "model input scale s_i (trailing-day mean + 0.5) vs training"),
], columns=["check", "group", "warn", "alert", "bad", "on_alert", "meaning"]).set_index("check")

CHECKS.to_csv(f"{OUT}/check_specification.csv")
CHECKS

,group,warn,alert,bad,on_alert,meaning
check,,,,,,
missing_fields,Schema,1.00,1.00,high,page,a required field is missing (e.g. renamed upst...
new_fields,Schema,1.00,inf,high,page,an unknown field appeared (usually the other h...
null_pct,Completeness,0.50,2.00,high,page,% of sessions with an empty required field (tr...
volume_ratio,Completeness,0.60,0.30,low,page,sessions this week / median of the last 4 good...
dead_weekdays,Completeness,1.00,2.00,high,page,working days with zero arrivals = ingestion ou...
invalid_pct,Validity,1.00,5.00,high,page,"% impossible sessions (ends before start, kWh ..."
duplicate_pct,Validity,0.01,2.00,high,page,% of session IDs received twice
new_ports,Categories,1.00,5.00,high,page,charging ports never seen before = capacity ch...
psi_vs_recent,Drift,0.10,0.25,high,page,occupancy distribution vs the last 4 good week...


Why these numbers:
* **PSI 0.10 / 0.25** are the standard cut-offs for a "moderate" and a "major" distribution shift.
* **Nulls, duplicates and invalid rows** are 0% in the training data, so even 1% is abnormal.
* **Volume** is compared with the *last 4 weeks*, not with training: 2021 has far fewer sessions than 2019, so a
  training baseline would alert every week.
* **Daily-curve correlation** between two normal weeks is about 0.95 to 0.99; a clock bug makes it collapse.

## 2. Load the raw sessions

We check the **raw** records, before cleaning. Cleaning silently drops bad rows, so problems like duplicates or
empty fields are only visible at this stage.

In [3]:
def load_raw(site):
    frames = []
    for path in sorted(glob.glob(f"{DATA_DIR}/acn_{site}_*.json")):
        with open(path) as f:
            items = json.load(f)
        if items:                                  # files after Sept 2021 are empty
            frames.append(pd.DataFrame(items))
    return pd.concat(frames, ignore_index=True)

raw = {site: load_raw(site) for site in SITES}
{site: len(df) for site, df in raw.items()}

{'caltech': 16126, 'jpl': 28863}

## 3. Rebuild occupancy exactly like the training pipeline

The cleaning rules and the occupancy formula are copied from `data_audit.ipynb`
(occupancy = number of cars plugged in per 15-minute step).

In [4]:
def clean_sessions(df):
    # Same rules as data_audit.ipynb: dedupe, drop impossible and stuck sessions
    df = df.copy()
    for col in ["connectionTime", "disconnectTime"]:
        if col not in df:
            df[col] = None
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    df["kWhDelivered"] = pd.to_numeric(df["kWhDelivered"], errors="coerce")
    hours = (df["disconnectTime"] - df["connectionTime"]).dt.total_seconds() / 3600
    stuck = (hours > 72) & (df["kWhDelivered"] < 0.5)
    keep = df["connectionTime"].notna() & df["disconnectTime"].notna() & (hours > 0) & ~stuck
    return df[keep].drop_duplicates("sessionID")


def occupancy(sessions, start=None, end=None):
    # +1 when a car arrives (rounded down), -1 when it leaves (rounded up), then running total
    arrivals = sessions["connectionTime"].dt.floor("15min").value_counts()
    departures = sessions["disconnectTime"].dt.ceil("15min").value_counts()
    level = arrivals.subtract(departures, fill_value=0).sort_index().cumsum()
    start = level.index[0] if start is None else start
    end = level.index[-1] if end is None else end
    grid = pd.date_range(start, end, freq="15min")
    return level.reindex(level.index.union(grid)).ffill().fillna(0).reindex(grid)

Sanity check: the rebuilt occupancy must equal every test target saved by `data_audit.ipynb`.

In [5]:
occ, split = {}, {}
for site in SITES:
    d = np.load(f"outputs/{site}_occupancy_data.npz")
    ts = pd.to_datetime(d["timestamps"])
    occ[site] = occupancy(clean_sessions(raw[site]))
    split[site] = {
        "train_start": ts[d["train_origins"][0] - L], "train_end": ts[d["train_origins"][-1] + H - 1],
        "test_start": ts[d["test_origins"][0]],      "test_end": ts[d["test_origins"][-1] + H - 1],
    }
    test_steps = d["test_origins"][:, None] + np.arange(H)
    max_diff = np.abs(occ[site].to_numpy()[test_steps] - d["test_y"]).max()
    print(f"{site:8s} max difference with training targets: {max_diff}")
    assert max_diff == 0

pd.DataFrame({site: {k: f"{t:%Y-%m-%d}" for k, t in dates.items()} for site, dates in split.items()}).T

caltech  max difference with training targets: 0.0


jpl      max difference with training targets: 0.0


,train_start,train_end,test_start,test_end
caltech,2019-01-01,2021-01-04,2021-05-10,2021-09-14
jpl,2019-01-02,2020-12-19,2021-04-26,2021-09-14


## 4. The reference: what "normal" looked like during training

We freeze a few facts about each site's **training period**: the occupancy values (with the long empty gaps removed,
as in training), the model scale s_i, the list of known charging ports, and the daily occupancy curve.
In production this file is saved next to the model, so a rollback restores the matching reference.

In [6]:
def not_in_long_gap(values, min_days=5):
    # False inside runs of zero occupancy lasting 5+ days (no data), as data_audit.ipynb does
    zero = np.r_[0, (values == 0).astype(int), 0]
    edges = np.flatnonzero(np.diff(zero))
    keep = np.ones(len(values), dtype=bool)
    for start, end in zip(edges[::2], edges[1::2]):
        if end - start >= min_days * 96:
            keep[start:end] = False
    return keep


def daily_curve(series):
    # Mean occupancy for each hour of the day (local time)
    local = series.tz_convert(TZ)
    return local.groupby(local.index.hour).mean().reindex(range(24)).to_numpy()


ref = {}
for site in SITES:
    s = split[site]
    train_occ = occ[site][s["train_start"]:s["train_end"]]
    train_scale = (occ[site].rolling(96, min_periods=1).mean() + 0.5)[s["train_start"]:s["train_end"]]
    keep = not_in_long_gap(train_occ.to_numpy())
    conn = pd.to_datetime(raw[site]["connectionTime"], utc=True)
    in_train = (conn >= s["train_start"]) & (conn <= s["train_end"])
    ref[site] = {
        "occupancy": train_occ.to_numpy()[keep],
        "scale": train_scale.to_numpy()[keep],
        "ports": sorted(raw[site].loc[in_train, "spaceID"].unique()),
        "daily_curve": daily_curve(train_occ),
    }

summary = {site: {"mean occupancy": round(float(r["occupancy"].mean()), 2),
                  "std occupancy": round(float(r["occupancy"].std()), 2),
                  "known ports": len(r["ports"])} for site, r in ref.items()}

for site, r in ref.items():
    with open(f"{OUT}/reference_{site}.json", "w") as f:
        json.dump({**summary[site], "ports": r["ports"],
                   "daily_curve": np.round(r["daily_curve"], 3).tolist()}, f, indent=2)
pd.DataFrame(summary)

,caltech,jpl
mean occupancy,5.46,11.38
std occupancy,8.34,18.39
known ports,55.00,52.00


## 5. Drift measures

Two simple ways to say *"how different is this week from the reference?"*:

* **PSI (Population Stability Index).** Cut the reference into 10 bins of equal size, count what share of the new
  week falls in each bin, and add up `(new - ref) * ln(new / ref)`. 0 means identical; above 0.25 is a major shift.
* **Wasserstein distance.** The average gap between matching percentiles of the two distributions, **in cars**.
  Easy to explain: "occupancy moved by about 2 cars".

In [7]:
def psi(reference, current, n_bins=10):
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, n_bins + 1)[1:-1]))
    ref_share = np.bincount(np.searchsorted(edges, reference), minlength=len(edges) + 1) / len(reference)
    cur_share = np.bincount(np.searchsorted(edges, current), minlength=len(edges) + 1) / len(current)
    ref_share, cur_share = np.clip(ref_share, 1e-4, None), np.clip(cur_share, 1e-4, None)  # avoid log(0)
    return float(np.sum((cur_share - ref_share) * np.log(cur_share / ref_share)))


def wasserstein(reference, current):
    q = np.linspace(0, 1, 101)
    return float(np.mean(np.abs(np.quantile(reference, q) - np.quantile(current, q))))


# Quick demo: identical data vs. data shifted by +3 cars
x = ref["caltech"]["occupancy"]
print(f"PSI same data: {psi(x, x):.3f} | PSI +3 cars: {psi(x, x + 3):.2f} | Wasserstein +3 cars: {wasserstein(x, x + 3):.1f}")

PSI same data: 0.000 | PSI +3 cars: 5.47 | Wasserstein +3 cars: 3.0


## 6. The weekly check

`check_week` takes one week of raw sessions and returns every metric from the table in section 1.
`status` turns a value into OK / WARN / ALERT.

In [8]:
def check_week(week_start, week_raw, week_occ, week_scale, recent_occ, recent_counts, ref_site, known_ports):
    m = {"week": week_start.tz_convert(TZ).strftime("%Y-%m-%d"), "sessions": len(week_raw)}

    # Schema: are the expected fields there?
    filled = [c for c in week_raw.columns if week_raw[c].notna().any()]
    m["missing_fields"] = sum(c not in filled for c in REQUIRED)
    m["new_fields"] = sum(c not in KNOWN for c in filled)

    # Completeness: is anything missing?
    m["null_pct"] = 100 * week_raw.reindex(columns=REQUIRED).isna().any(axis=1).mean()
    m["volume_ratio"] = len(week_raw) / np.median(recent_counts)
    arrival_days = set(pd.to_datetime(week_raw["connectionTime"], utc=True, errors="coerce")
                       .dropna().dt.tz_convert(TZ).dt.date)
    days = [(week_start + pd.Timedelta(days=i)).tz_convert(TZ).date() for i in range(7)]
    workdays = [d for d in days if d.weekday() < 5 and d not in CA_HOLIDAYS]
    m["dead_weekdays"] = sum(d not in arrival_days for d in workdays)

    # Validity: are the values possible?
    start = pd.to_datetime(week_raw["connectionTime"], utc=True, errors="coerce")
    end = pd.to_datetime(week_raw.get("disconnectTime"), utc=True, errors="coerce")
    kwh = pd.to_numeric(week_raw["kWhDelivered"], errors="coerce")
    m["invalid_pct"] = 100 * ((end <= start) | (kwh < 0) | (kwh > 100)).mean()
    m["duplicate_pct"] = 100 * week_raw["sessionID"].duplicated().mean()

    # Categories: any charging port we have never seen?
    new = set(week_raw["spaceID"].dropna()) - known_ports
    m["new_ports"] = len(new)
    known_ports |= new      # the on-call registers them, so they alert only once

    # Drift
    m["mean_occupancy"] = week_occ.mean()
    m["psi_vs_recent"] = psi(recent_occ.to_numpy(), week_occ.to_numpy())
    now, before = daily_curve(week_occ), daily_curve(recent_occ)
    m["hour_corr"] = np.corrcoef(now, before)[0, 1] if now.std() > 0 else np.nan
    m["psi_vs_train"] = psi(ref_site["occupancy"], week_occ.to_numpy())
    m["psi_scale"] = psi(ref_site["scale"], week_scale.to_numpy())
    m["wasserstein_cars"] = wasserstein(ref_site["occupancy"], week_occ.to_numpy())
    return m


def status(value, warn, alert, bad):
    if pd.isna(value):
        return "n/a"
    if bad == "high":
        return "ALERT" if value >= alert else "WARN" if value >= warn else "OK"
    return "ALERT" if value <= alert else "WARN" if value <= warn else "OK"

## 7. Replay the test period week by week

For each week the loop:
1. takes the raw sessions that arrived that week,
2. rebuilds occupancy from the cleaned sessions,
3. runs `check_week` and computes the statuses,
4. decides whether to **page** (worst status among the `page` checks).

**Quarantine.** "The last 4 weeks" means the last 4 **good** weeks. A week with broken data (an ALERT on a Schema,
Completeness or Validity check, or a broken daily curve) is kept out, otherwise one broken week would make the next
four healthy weeks look abnormal. Real changes (new ports, drift) are accepted and become the new normal.

In [9]:
PAGE_CHECKS = CHECKS.index[CHECKS["on_alert"] == "page"]
DATA_ERROR_CHECKS = list(CHECKS.index[CHECKS["group"].isin(["Schema", "Completeness", "Validity"])]) + ["hour_corr"]
SEVERITY = {"n/a": 0, "OK": 0, "WARN": 1, "ALERT": 2}


def between(series, start, end):
    return series[(series.index >= start) & (series.index < end)]


def replay(site, raw_df):
    s = split[site]
    sessions = clean_sessions(raw_df)
    site_occ = occupancy(sessions, start=occ[site].index[0], end=s["test_end"])
    site_scale = site_occ.rolling(96, min_periods=1).mean() + 0.5
    arrived = pd.to_datetime(raw_df["connectionTime"], utc=True, errors="coerce")
    known_ports = set(ref[site]["ports"])

    # Weeks start on Monday 00:00 local time
    start_local = s["test_start"].tz_convert(TZ)
    first = (start_local - pd.Timedelta(days=start_local.dayofweek)).normalize()
    first = first + WEEK if first < start_local else first
    weeks = [first + k * WEEK for k in range(int((s["test_end"] - first) / WEEK))]

    good = [first - k * WEEK for k in (4, 3, 2, 1)]           # the 4 weeks before the replay
    counts = {w: int(((arrived >= w) & (arrived < w + WEEK)).sum()) for w in good}

    rows = []
    for w in weeks:
        last4 = good[-4:]
        week_raw = raw_df[((arrived >= w) & (arrived < w + WEEK)).to_numpy()]
        recent_occ = pd.concat([between(site_occ, g, g + WEEK) for g in last4])
        m = check_week(w, week_raw, between(site_occ, w, w + WEEK), between(site_scale, w, w + WEEK),
                       recent_occ, [counts[g] for g in last4], ref[site], known_ports)
        for check, c in CHECKS.iterrows():
            m[f"s_{check}"] = status(m[check], c["warn"], c["alert"], c["bad"])
        m["page"] = max((m[f"s_{c}"] for c in PAGE_CHECKS), key=SEVERITY.get)
        m["fired"] = ", ".join(c for c in PAGE_CHECKS if m[f"s_{c}"] in ("WARN", "ALERT"))
        m["quarantined"] = any(m[f"s_{c}"] == "ALERT" for c in DATA_ERROR_CHECKS)
        m["reference_weeks"] = last4
        if not m["quarantined"]:
            good.append(w)
            counts[w] = len(week_raw)
        rows.append(m)
    return pd.DataFrame(rows), site_occ


SHOW = ["sessions", "volume_ratio", "mean_occupancy", "psi_vs_train", "psi_vs_recent",
        "hour_corr", "page", "quarantined", "fired"]

## 8. Run A: replay the real data

In [10]:
run_a = {site: replay(site, raw[site]) for site in SITES}
for site, (res, _) in run_a.items():
    res.drop(columns="reference_weeks").to_csv(f"{OUT}/weekly_clean_{site}.csv", index=False)

run_a["caltech"][0].set_index("week")[SHOW].round(2)

,sessions,volume_ratio,mean_occupancy,psi_vs_train,psi_vs_recent,hour_corr,page,quarantined,fired
week,,,,,,,,,
2021-05-17,79,1.08,1.42,1.42,0.02,0.83,WARN,False,hour_corr
2021-05-24,89,1.16,1.61,1.40,0.03,0.98,OK,False,
2021-05-31,88,1.10,1.53,1.43,0.09,0.94,OK,False,
2021-06-07,99,1.17,1.68,1.39,0.02,0.95,OK,False,
2021-06-14,103,1.16,1.80,1.43,0.10,0.95,OK,False,
2021-06-21,105,1.12,1.78,0.98,0.09,0.97,OK,False,
2021-06-28,108,1.07,2.18,1.48,0.11,0.97,WARN,False,psi_vs_recent
2021-07-05,85,0.82,1.74,1.40,0.07,0.96,OK,False,
2021-07-12,92,0.88,2.53,0.96,0.21,0.97,WARN,False,psi_vs_recent


In [11]:
run_a["jpl"][0].set_index("week")[SHOW].round(2)

,sessions,volume_ratio,mean_occupancy,psi_vs_train,psi_vs_recent,hour_corr,page,quarantined,fired
week,,,,,,,,,
2021-04-26,163,1.06,5.48,0.87,0.14,0.99,WARN,False,psi_vs_recent
2021-05-03,142,0.90,5.33,0.61,0.02,0.99,OK,False,
2021-05-10,166,1.06,5.43,0.98,0.11,1.00,WARN,False,psi_vs_recent
2021-05-17,160,1.00,6.04,0.43,0.09,1.00,OK,False,
2021-05-24,185,1.15,6.66,0.51,0.05,1.00,OK,False,
2021-05-31,127,0.78,4.28,0.57,0.06,1.00,OK,False,
2021-06-07,190,1.17,7.12,0.46,0.11,1.00,WARN,False,psi_vs_recent
2021-06-14,168,0.97,6.08,0.48,0.01,1.00,OK,False,
2021-06-21,201,1.14,7.11,0.41,0.08,0.99,OK,False,


In [12]:
def count_statuses(res):
    return pd.Series({
        "weeks": len(res),
        "paged (ALERT)": (res["page"] == "ALERT").sum(),
        "warned only": (res["page"] == "WARN").sum(),
        "retraining ticket open (PSI vs training ALERT)": (res["s_psi_vs_train"] == "ALERT").sum(),
    })

pd.DataFrame({site: count_statuses(res) for site, (res, _) in run_a.items()})

,caltech,jpl
weeks,17,20
paged (ALERT),1,3
warned only,6,9
retraining ticket open (PSI vs training ALERT),17,20


**What run A shows**

* **The data has drifted far from training at both sites.** PSI vs training is above 0.25 in every single week
  (17/17 at Caltech, 20/20 at JPL): training includes pre-COVID 2019, when the sites were much busier. This opens
  the retraining ticket (it does not page), and it is consistent with the Caltech intervals covering less than 80%
  in Milestone 2.
* **Few false alarms.** Caltech pages once in 17 weeks (week of 16 Aug, a jump in occupancy as usage climbs through
  August). JPL pages 3 times in 20 weeks, all around the real incident below. WARNs (6 and 9) come mostly from
  week-to-week noise and only show in the status board.
* **JPL has a real incident in July 2021**, found without any help. Next cell.

In [13]:
start, end = pd.Timestamp("2021-07-12", tz=TZ), pd.Timestamp("2021-08-02", tz=TZ)
arrived = pd.to_datetime(raw["jpl"]["connectionTime"], utc=True).dt.tz_convert(TZ)
arrived = arrived[(arrived >= start) & (arrived < end)]
jpl_occ = occ["jpl"].tz_convert(TZ)[start:end - pd.Timedelta("15min")]

incident = pd.DataFrame({
    "sessions arrived": arrived.dt.date.value_counts(),
    "max cars plugged in": jpl_occ.groupby(jpl_occ.index.date).max(),
    "min cars plugged in": jpl_occ.groupby(jpl_occ.index.date).min(),
}).fillna(0).astype(int).sort_index()
incident.insert(0, "weekday", [pd.Timestamp(d).day_name()[:3] for d in incident.index])
incident.index.name = "JPL, day"
print(incident.to_string())

print("\nWhat the monitor reported for those two weeks:")
print(run_a["jpl"][0].set_index("week").loc[["2021-07-19", "2021-07-26"],
      ["sessions", "volume_ratio", "dead_weekdays", "hour_corr", "page", "fired"]].round(2).to_string())

           weekday  sessions arrived  max cars plugged in  min cars plugged in
JPL, day                                                                      
2021-07-12     Mon                48                   37                    0
2021-07-13     Tue                46                   28                    0
2021-07-14     Wed                41                   28                    0
2021-07-15     Thu                44                   34                    0
2021-07-16     Fri                11                    9                    0
2021-07-17     Sat                 0                    8                    8
2021-07-18     Sun                 0                    8                    8
2021-07-19     Mon                27                   29                    0
2021-07-20     Tue                20                   18                    0
2021-07-21     Wed                32                   32                   14
2021-07-22     Thu                 0                

Sessions stay open overnight, then all close at once, then there are **six days without a single record**. That looks
like a logging outage. The offline gap filter in `data_audit.ipynb` removed most of these windows, so the Milestone 2
test scores are barely affected. **In production there is no offline filter**: without this check, the service would
have published six days of forecasts based on missing data.

## 9. Run B: break the data on purpose (Caltech)

Each function below simulates one realistic production problem in one week. There is a normal week between
two problems, so every alert can be traced back to its cause. The new chargers stay until the end.

In [14]:
def rows_in_week(df, week):
    t = pd.to_datetime(df["connectionTime"], utc=True)
    return ((t >= week) & (t < week + WEEK)).to_numpy()

def outage(df, week):
    # Tue, Wed and Thu are lost by the ingestion job
    day = pd.to_datetime(df["connectionTime"], utc=True).dt.tz_convert(TZ).dt.dayofweek
    return df[~(rows_in_week(df, week) & day.isin([1, 2, 3]).to_numpy())]

def null_burst(df, week):
    # 15% of sessions arrive without a disconnect time
    df = df.copy()
    rows = df.index[rows_in_week(df, week)]
    df.loc[rows[::7], "disconnectTime"] = None
    return df

def duplicates(df, week):
    # The producer re-sends 25% of the sessions
    return pd.concat([df, df[rows_in_week(df, week)].iloc[::4]], ignore_index=True)

def schema_rename(df, week):
    # Upstream API renames disconnectTime -> disconnect_time
    df = df.copy()
    rows = rows_in_week(df, week)
    df.loc[rows, "disconnect_time"] = df.loc[rows, "disconnectTime"]
    df.loc[rows, "disconnectTime"] = None
    return df

def clock_bug(df, week):
    # Local times are stored as if they were UTC: everything is shifted by +8 hours
    df = df.copy()
    rows = rows_in_week(df, week)
    for col in ["connectionTime", "disconnectTime"]:
        shifted = pd.to_datetime(df.loc[rows, col], utc=True) + pd.Timedelta(hours=8)
        df.loc[rows, col] = shifted.astype(str)
    return df

def corrupt_records(df, week):
    # Firmware bug: 8% of sessions report negative energy
    df = df.copy()
    rows = df.index[rows_in_week(df, week)]
    df.loc[rows[::12], "kWhDelivered"] = -1.0
    return df

def new_chargers(df, week, n=20):
    # 20 new ports from this week on: they copy the usage of existing ports
    after = df[pd.to_datetime(df["connectionTime"], utc=True) >= week]
    ports = sorted(after["spaceID"].unique())[:n]
    copies = []
    for k, port in enumerate(ports):
        c = after[after["spaceID"] == port].copy()
        c["spaceID"] = f"NEW-{k + 1:02d}"
        c["sessionID"] = c["sessionID"] + f"-new{k + 1}"
        copies.append(c)
    return pd.concat([df, *copies], ignore_index=True)

In [15]:
weeks = pd.to_datetime(run_a["caltech"][0]["week"]).dt.tz_localize(TZ)

PROBLEMS = [  # (week number, function, check(s) that should catch it)
    (2,  outage,          "dead_weekdays"),
    (4,  null_burst,      "null_pct"),
    (6,  duplicates,      "duplicate_pct"),
    (8,  schema_rename,   "missing_fields"),
    (10, clock_bug,       "hour_corr"),
    (12, corrupt_records, "invalid_pct"),
    (14, new_chargers,    "new_ports"),
]

broken = raw["caltech"]
for week_no, inject, _ in PROBLEMS:
    broken = inject(broken, weeks[week_no])

run_b, run_b_occ = replay("caltech", broken)
run_b.drop(columns="reference_weeks").to_csv(f"{OUT}/weekly_injected_caltech.csv", index=False)
run_b.set_index("week")[SHOW].round(2)

,sessions,volume_ratio,mean_occupancy,psi_vs_train,psi_vs_recent,hour_corr,page,quarantined,fired
week,,,,,,,,,
2021-05-17,79,1.08,1.42,1.42,0.02,0.83,WARN,False,hour_corr
2021-05-24,89,1.16,1.61,1.40,0.03,0.98,OK,False,
2021-05-31,43,0.54,0.77,1.66,0.30,0.86,ALERT,True,"volume_ratio, dead_weekdays, psi_vs_recent, ho..."
2021-06-07,99,1.24,1.68,1.39,0.03,0.97,OK,False,
2021-06-14,103,1.21,1.64,1.42,0.04,0.93,ALERT,True,null_pct
2021-06-21,105,1.24,1.78,0.98,0.09,0.98,OK,False,
2021-06-28,135,1.44,2.18,1.48,0.17,0.94,ALERT,True,"duplicate_pct, psi_vs_recent"
2021-07-05,85,0.90,1.74,1.40,0.08,0.92,OK,False,
2021-07-12,92,0.98,0.00,5.89,5.13,NaN,ALERT,True,"missing_fields, new_fields, null_pct, psi_vs_r..."


### Did every check catch its problem?

In [16]:
detection = pd.DataFrame([{
    "week": run_b.loc[w, "week"],
    "problem injected": f.__name__,
    "check that should fire": expected,
    "its status": run_b.loc[w, f"s_{expected}"],
    "paged?": run_b.loc[w, "page"],
    "same week in run A": run_a["caltech"][0].loc[w, "page"],
    "all checks that fired": run_b.loc[w, "fired"],
} for w, f, expected in PROBLEMS])

detection.to_csv(f"{OUT}/detection_matrix.csv", index=False)
print(f"Caught {(detection['its status'] == 'ALERT').sum()} of {len(detection)} problems")
detection

Caught 7 of 7 problems


,week,problem injected,check that should fire,its status,paged?,same week in run A,all checks that fired
0,2021-05-31,outage,dead_weekdays,ALERT,ALERT,OK,"volume_ratio, dead_weekdays, psi_vs_recent, ho..."
1,2021-06-14,null_burst,null_pct,ALERT,ALERT,OK,null_pct
2,2021-06-28,duplicates,duplicate_pct,ALERT,ALERT,WARN,"duplicate_pct, psi_vs_recent"
3,2021-07-12,schema_rename,missing_fields,ALERT,ALERT,WARN,"missing_fields, new_fields, null_pct, psi_vs_r..."
4,2021-07-26,clock_bug,hour_corr,ALERT,ALERT,OK,hour_corr
5,2021-08-09,corrupt_records,invalid_pct,ALERT,ALERT,WARN,"invalid_pct, psi_vs_recent"
6,2021-08-23,new_chargers,new_ports,ALERT,ALERT,OK,"new_ports, psi_vs_recent"


Two things to notice:

* **The usual cleaning would hide most of these problems.** Duplicates, empty fields and impossible values are
  dropped by the `data_audit` rules, so occupancy looks almost normal. Only the checks on the raw records see them.
* **New chargers are a real change, not an error.** The ports are registered after their first alert and the
  weeks are accepted as normal (not quarantined). The drift check keeps paging until the 4-week reference is made
  only of weeks with the new chargers, so in practice the on-call acknowledges the change after the first page.

## 10. Status board and drift statistics

This is the dashboard in text form. **Status board:** one row per check, one column per week:
`ok`, `W` (WARN) or `A` (ALERT); `-` means it could not be computed that week. The last rows say whether someone was
paged and whether the week was quarantined. **Drift statistics:** the actual numbers behind the drift checks.

In [17]:
SHORT = {"OK": "ok", "WARN": "W", "ALERT": "A", "n/a": "-"}


def status_board(res, labels=None):
    board = res.set_index("week")[[f"s_{c}" for c in CHECKS.index] + ["page"]].replace(SHORT)
    board.columns = [f"{c} ({CHECKS.loc[c, 'on_alert']})" for c in CHECKS.index] + ["PAGED?"]
    board["quarantined"] = res.set_index("week")["quarantined"].map({True: "Q", False: ""})
    board = board.T
    board.columns = [w[5:] + (f" {labels[i]}" if labels and i in labels else "") for i, w in enumerate(res["week"])]
    return board


DRIFT = {"mean_occupancy": "mean cars", "psi_vs_train": "PSI vs training", "psi_vs_recent": "PSI vs last 4 wks",
         "psi_scale": "PSI of s_i", "wasserstein_cars": "Wasserstein (cars)", "hour_corr": "daily-curve corr."}


def drift_table(res):
    return res.set_index("week")[list(DRIFT)].rename(columns=DRIFT).round(2)

In [18]:
print("Caltech, run B (injected problems; the problem name is next to its week)\n")
print(status_board(run_b, {w: f.__name__ for w, f, _ in PROBLEMS}).to_string())

Caltech, run B (injected problems; the problem name is next to its week)

                      05-17 05-24 05-31 outage 06-07 06-14 null_burst 06-21 06-28 duplicates 07-05 07-12 schema_rename 07-19 07-26 clock_bug 08-02 08-09 corrupt_records 08-16 08-23 new_chargers 08-30 09-06
missing_fields (page)    ok    ok           ok    ok               ok    ok               ok    ok                   A    ok              ok    ok                    ok    ok                 ok    ok    ok
new_fields (page)        ok    ok           ok    ok               ok    ok               ok    ok                   W    ok              ok    ok                    ok    ok                 ok    ok    ok
null_pct (page)          ok    ok           ok    ok                A    ok               ok    ok                   A    ok              ok    ok                    ok    ok                 ok    ok    ok
volume_ratio (page)      ok    ok            W    ok               ok    ok               ok    ok    

In [19]:
print("JPL, run A (real data)\n")
print(status_board(run_a["jpl"][0]).to_string())

JPL, run A (real data)

                      04-26 05-03 05-10 05-17 05-24 05-31 06-07 06-14 06-21 06-28 07-05 07-12 07-19 07-26 08-02 08-09 08-16 08-23 08-30 09-06
missing_fields (page)    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
new_fields (page)        ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
null_pct (page)          ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
volume_ratio (page)      ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok     W     W    ok    ok    ok    ok    ok    ok
dead_weekdays (page)     ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok     A     A    ok    ok    ok    ok    ok    ok
invalid_pct (page)       ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok     W    ok    ok    ok    ok 

In [20]:
print("Caltech, run A (real data)\n")
print(status_board(run_a["caltech"][0]).to_string())

Caltech, run A (real data)

                      05-17 05-24 05-31 06-07 06-14 06-21 06-28 07-05 07-12 07-19 07-26 08-02 08-09 08-16 08-23 08-30 09-06
missing_fields (page)    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
new_fields (page)        ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
null_pct (page)          ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
volume_ratio (page)      ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
dead_weekdays (page)     ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
invalid_pct (page)       ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok
duplicate_pct (page)     ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    ok    o

In [21]:
print("Drift statistics, Caltech, run A\n")
print(drift_table(run_a["caltech"][0]).to_string())
print("\nDrift statistics, JPL, run A\n")
print(drift_table(run_a["jpl"][0]).to_string())

Drift statistics, Caltech, run A

            mean cars  PSI vs training  PSI vs last 4 wks  PSI of s_i  Wasserstein (cars)  daily-curve corr.
week                                                                                                        
2021-05-17       1.42             1.42               0.02        5.94                4.20               0.83
2021-05-24       1.61             1.40               0.03        5.16                4.01               0.98
2021-05-31       1.53             1.43               0.09        6.66                4.11               0.94
2021-06-07       1.68             1.39               0.02        5.84                3.92               0.95
2021-06-14       1.80             1.43               0.10        5.85                3.83               0.95
2021-06-21       1.78             0.98               0.09        5.16                3.83               0.97
2021-06-28       2.18             1.48               0.11        6.06                3.56     

## 11. Alert log and what the on-call does

Every ALERT on a paging check becomes one line, with the first action from the runbook. While a data alert is open,
the forecast API serves the **labelled fallback** (persistence or seasonal naive) defined in Milestone 2.

In [22]:
ACTION = {
    "Schema":       "Page data engineering: pause the model, serve the fallback, fix the field mapping, backfill.",
    "Completeness": "Page data engineering: check the ingestion job, serve the fallback, backfill the missing days.",
    "Validity":     "Ticket to data engineering: cleaning drops the bad rows; find the source (firmware, retries).",
    "Categories":   "Page ML on-call: register the new ports, check interval coverage, plan recalibration.",
    "Drift":        "ML on-call: holiday? clock bug? If it lasts 2+ weeks with clean data, plan retraining.",
}

alerts = []
for run, site, res in [("B injected", "caltech", run_b),
                       ("A real", "caltech", run_a["caltech"][0]),
                       ("A real", "jpl", run_a["jpl"][0])]:
    for _, row in res.iterrows():
        for check in PAGE_CHECKS:
            if row[f"s_{check}"] == "ALERT":
                alerts.append({"run": run, "site": site, "week": row["week"], "check": check,
                               "value": round(row[check], 3), "threshold": CHECKS.loc[check, "alert"],
                               "action": ACTION[CHECKS.loc[check, "group"]]})

alerts = pd.DataFrame(alerts)
alerts.to_csv(f"{OUT}/alert_log.csv", index=False)
alerts

,run,site,week,check,value,threshold,action
0,B injected,caltech,2021-05-31,dead_weekdays,3.000,2.00,Page data engineering: check the ingestion job...
1,B injected,caltech,2021-05-31,psi_vs_recent,0.301,0.25,ML on-call: holiday? clock bug? If it lasts 2+...
2,B injected,caltech,2021-06-14,null_pct,14.563,2.00,Page data engineering: check the ingestion job...
3,B injected,caltech,2021-06-28,duplicate_pct,20.000,2.00,Ticket to data engineering: cleaning drops the...
4,B injected,caltech,2021-07-12,missing_fields,1.000,1.00,"Page data engineering: pause the model, serve ..."
5,B injected,caltech,2021-07-12,null_pct,100.000,2.00,Page data engineering: check the ingestion job...
6,B injected,caltech,2021-07-12,psi_vs_recent,5.131,0.25,ML on-call: holiday? clock bug? If it lasts 2+...
7,B injected,caltech,2021-07-26,hour_corr,-0.432,0.75,ML on-call: holiday? clock bug? If it lasts 2+...
8,B injected,caltech,2021-08-09,invalid_pct,8.451,5.00,Ticket to data engineering: cleaning drops the...
9,B injected,caltech,2021-08-16,psi_vs_recent,0.323,0.25,ML on-call: holiday? clock bug? If it lasts 2+...


## 12. Summary

| | Result |
|---|---|
| Rebuilt occupancy vs training targets | identical (difference 0) |
| Injected problems caught (run B) | 7 of 7, each with an ALERT in its own week |
| False alarms on real data (run A) | Caltech 1 page in 17 weeks; JPL 3 in 20, all around the real incident |
| Real incident found (run A) | JPL, 22–27 July 2021: six days with no records |
| Regime drift vs training (run A) | PSI above 0.25 every week at both sites → retraining ticket |

**Files written to `outputs/monitoring/`:** `check_specification.csv`, `reference_<site>.json`, `weekly_clean_<site>.csv`,
`weekly_injected_caltech.csv`, `detection_matrix.csv` and `alert_log.csv`.

**Next:** the model-quality notebook (rolling MAE vs seasonal naive, 80% interval coverage, 3-day rollback rule)
reuses the same weekly replay.